# Caso Práctico Integrador — Detección de Transacciones Anómalas
## Presentan Diego Arturo Sánchez López | Uriel Márquez García

**Objetivo:** Construir un modelo de Machine Learning para detectar transacciones fraudulentas en un dataset con fuerte desbalance de clases (~2% fraude).

**random_state = 42** fijado en todos los procesos estocásticos.

In [ ]:
# ──────────────────────────────────────────────
# Importaciones
# ──────────────────────────────────────────────
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    roc_curve, precision_recall_curve, f1_score, accuracy_score,
    precision_score, recall_score
)
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
import shap

SEED = 42
np.random.seed(SEED)
print('Librerías cargadas correctamente.')

---
## MÓDULO 1 — Análisis Exploratorio de Datos (EDA)

In [ ]:
# Carga del dataset
df = pd.read_excel('base.xlsx')
print(f'Dimensiones: {df.shape}')
df.head()

In [ ]:
# Estadísticos descriptivos
df.describe().T

In [ ]:
# Tipos de datos y valores nulos
print('Tipos de datos:')
print(df.dtypes)
print(f'\nValores nulos por columna:')
print(df.isnull().sum())

In [ ]:
# Distribución de la variable objetivo — desbalance de clases
counts = df['fraud'].value_counts()
print('Distribución de la variable objetivo:')
print(counts)
print(f'\nTasa de fraude: {counts[1]/len(df)*100:.2f}%')

fig, ax = plt.subplots(figsize=(5, 3))
ax.bar(['Legítima (0)', 'Fraude (1)'], counts.values, color=['#065A82', '#F96167'])
ax.set_title('Distribución de Clases')
ax.set_ylabel('Cantidad de transacciones')
for i, v in enumerate(counts.values):
    ax.text(i, v + 50, f'{v:,}\n({v/len(df)*100:.1f}%)', ha='center', fontsize=9)
plt.tight_layout()
plt.savefig('fig_desbalance.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# Análisis de outliers — boxplots en variables numéricas clave
num_cols = ['amount', 'client_credit_score', 'transaction_frequency',
            'annual_income', 'account_balance']

fig, axes = plt.subplots(1, len(num_cols), figsize=(14, 4))
for ax, col in zip(axes, num_cols):
    df.boxplot(column=col, ax=ax)
    ax.set_title(col, fontsize=8)
plt.suptitle('Boxplots — Detección de Outliers', y=1.02)
plt.tight_layout()
plt.savefig('fig_boxplots.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# Correlación entre variables numéricas
num_df = df.select_dtypes(include='number').drop(columns=['transaction_id'])
corr = num_df.corr()

fig, ax = plt.subplots(figsize=(10, 7))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', ax=ax, linewidths=0.5)
ax.set_title('Matriz de Correlación')
plt.tight_layout()
plt.savefig('fig_correlacion.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# Comportamiento diferenciado por clase (fraude vs legítima)
fig, axes = plt.subplots(2, 3, figsize=(14, 7))
vars_to_plot = ['amount', 'client_credit_score', 'transaction_frequency',
                'annual_income', 'account_balance', 'customer_age']

for ax, col in zip(axes.flatten(), vars_to_plot):
    df[df.fraud == 0][col].hist(ax=ax, alpha=0.6, bins=30, color='#065A82', label='Legítima')
    df[df.fraud == 1][col].hist(ax=ax, alpha=0.6, bins=30, color='#F96167', label='Fraude')
    ax.set_title(col, fontsize=9)
    ax.legend(fontsize=7)

plt.suptitle('Distribución por Clase: Fraude vs Legítima', fontsize=12)
plt.tight_layout()
plt.savefig('fig_por_clase.png', dpi=120, bbox_inches='tight')
plt.show()

**Hallazgos EDA:**
- El dataset tiene 10,000 registros, 17 variables, sin valores nulos.
- La tasa de fraude es del 2% (200 registros), generando un fuerte desbalance de clases (ratio 49:1).
- Las transacciones fraudulentas tienden a tener montos más altos y menor credit score.
- No se observan correlaciones extremas entre predictores que sugieran redundancia severa.

---
## MÓDULO 2 — Preparación y Selección de Variables

In [ ]:
# Extracción de características desde timestamp
df['hour']       = df['timestamp'].dt.hour
df['day_of_week'] = df['timestamp'].dt.dayofweek  # 0=Lun, 6=Dom
df['month']      = df['timestamp'].dt.month
df['is_weekend'] = df['day_of_week'].isin([5, 6]).astype(int)

print('Variables temporales creadas: hour, day_of_week, month, is_weekend')

In [ ]:
# Codificación de variables categóricas con Label Encoding
# Justificación: variables con pocas categorías ordinales o nominales de baja cardinalidad.
cat_cols = ['transaction_type', 'location', 'marital_status', 'housing_type']

df_model = df.copy()
le = LabelEncoder()
for col in cat_cols:
    df_model[col] = le.fit_transform(df_model[col])

# Variables a excluir: transaction_id (identificador), timestamp (ya procesado)
features = [c for c in df_model.columns if c not in ['transaction_id', 'timestamp', 'fraud']]
X = df_model[features]
y = df_model['fraud']

print(f'Features utilizadas ({len(features)}):\n{features}')

In [ ]:
# División train/test — ANTES de cualquier preprocesamiento (evitar data leakage)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)

print(f'Train: {X_train.shape[0]} registros | Test: {X_test.shape[0]} registros')
print(f'Fraude en train: {y_train.sum()} ({y_train.mean()*100:.1f}%)')
print(f'Fraude en test:  {y_test.sum()} ({y_test.mean()*100:.1f}%)')

**Decisiones de preparación:**
- Se extrajeron 4 variables temporales desde `timestamp` (hora, día, mes, fin de semana).
- Se aplicó Label Encoding a las 4 variables categóricas (baja cardinalidad).
- División 80/20 estratificada para preservar el 2% de fraude en ambos conjuntos.
- La normalización se aplica dentro del pipeline para evitar data leakage.

---
## MÓDULO 3 — Estrategia ante Desbalance y Construcción del Modelo

**Estrategia elegida:** SMOTE (Synthetic Minority Over-sampling Technique) aplicado **solo sobre el conjunto de entrenamiento** dentro del pipeline, combinado con `class_weight` en los modelos de árbol.

**Justificación:** Con solo 2% de fraude, un modelo sin tratamiento ignorará la clase minoritaria. SMOTE genera ejemplos sintéticos plausibles de la clase fraude, mejorando la capacidad de detección sin duplicar registros reales.

In [ ]:
# ── Modelo 1: Random Forest con SMOTE ──
rf_pipe = ImbPipeline([
    ('scaler', StandardScaler()),
    ('smote',  SMOTE(random_state=SEED)),
    ('clf',    RandomForestClassifier(random_state=SEED, class_weight='balanced'))
])

rf_params = {
    'clf__n_estimators':      [100, 200],
    'clf__max_depth':         [5, 10, None],
    'clf__min_samples_split': [2, 5]
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

rf_search = RandomizedSearchCV(
    rf_pipe, rf_params, n_iter=8, cv=cv,
    scoring='f1', n_jobs=-1, random_state=SEED, verbose=0
)
rf_search.fit(X_train, y_train)
print(f'RF — Mejor F1 CV: {rf_search.best_score_:.4f}')
print(f'Mejores parámetros: {rf_search.best_params_}')

In [ ]:
# ── Modelo 2: XGBoost con SMOTE ──
scale_pos = int((y_train == 0).sum() / (y_train == 1).sum())

xgb_pipe = ImbPipeline([
    ('scaler', StandardScaler()),
    ('smote',  SMOTE(random_state=SEED)),
    ('clf',    XGBClassifier(random_state=SEED, eval_metric='logloss',
                              scale_pos_weight=scale_pos))
])

xgb_params = {
    'clf__n_estimators':  [100, 200],
    'clf__max_depth':     [3, 5, 7],
    'clf__learning_rate': [0.05, 0.1]
}

xgb_search = RandomizedSearchCV(
    xgb_pipe, xgb_params, n_iter=8, cv=cv,
    scoring='f1', n_jobs=-1, random_state=SEED, verbose=0
)
xgb_search.fit(X_train, y_train)
print(f'XGB — Mejor F1 CV: {xgb_search.best_score_:.4f}')
print(f'Mejores parámetros: {xgb_search.best_params_}')

In [ ]:
# ── Modelo OPCIONAL: Isolation Forest (no supervisado) ──
# Supuesto: la variable 'fraud' no está disponible en entrenamiento.
scaler_iso = StandardScaler()
X_train_scaled = scaler_iso.fit_transform(X_train)
X_test_scaled  = scaler_iso.transform(X_test)

iso = IsolationForest(contamination=0.02, random_state=SEED)
iso.fit(X_train_scaled)
# Isolation Forest devuelve -1 (anomalía) y 1 (normal)
y_pred_iso = (iso.predict(X_test_scaled) == -1).astype(int)
print(f'Isolation Forest — Recall: {recall_score(y_test, y_pred_iso):.4f}')
print(f'Isolation Forest — F1:     {f1_score(y_test, y_pred_iso):.4f}')

---
## MÓDULO 4 — Evaluación y Comparación de Modelos

In [ ]:
def evaluar_modelo(nombre, modelo, X_test, y_test, umbral=0.5):
    """Calcula métricas completas para un modelo dado."""
    y_prob = modelo.predict_proba(X_test)[:, 1]
    y_pred = (y_prob >= umbral).astype(int)
    
    # KS statistic
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    ks = max(tpr - fpr)
    
    return {
        'Modelo':     nombre,
        'Accuracy':   accuracy_score(y_test, y_pred),
        'Precision':  precision_score(y_test, y_pred),
        'Recall':     recall_score(y_test, y_pred),
        'F1':         f1_score(y_test, y_pred),
        'AUC-ROC':    roc_auc_score(y_test, y_prob),
        'KS':         ks,
        '_prob':      y_prob,
        '_pred':      y_pred
    }

res_rf  = evaluar_modelo('Random Forest',    rf_search.best_estimator_,  X_test, y_test)
res_xgb = evaluar_modelo('XGBoost',          xgb_search.best_estimator_, X_test, y_test)

# Tabla comparativa
cols = ['Modelo', 'Accuracy', 'Precision', 'Recall', 'F1', 'AUC-ROC', 'KS']
tabla = pd.DataFrame([res_rf, res_xgb])[cols]
tabla.set_index('Modelo', inplace=True)
tabla = tabla.applymap(lambda x: f'{x:.4f}')
print('=== Tabla Comparativa de Modelos ===')
tabla

In [ ]:
# Selección del modelo final basada en F1 y AUC-ROC
# XGBoost suele ser superior en datasets desbalanceados; usamos el de mayor F1.
mejor_res = max([res_rf, res_xgb], key=lambda x: x['F1'])
mejor_modelo = rf_search.best_estimator_ if 'Random' in mejor_res['Modelo'] else xgb_search.best_estimator_
print(f'Modelo seleccionado: {mejor_res["Modelo"]}')

# Matriz de confusión
cm = confusion_matrix(y_test, mejor_res['_pred'])
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=['Pred Legítima', 'Pred Fraude'],
            yticklabels=['Real Legítima', 'Real Fraude'])
ax.set_title(f'Matriz de Confusión — {mejor_res["Modelo"]}')
plt.tight_layout()
plt.savefig('fig_confusion.png', dpi=120, bbox_inches='tight')
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f'\nVerdaderos Positivos (TP = fraudes detectados): {tp}')
print(f'Falsos Negativos (FN = fraudes no detectados):  {fn}  ← riesgo operacional')
print(f'Falsos Positivos (FP = bloqueos innecesarios):  {fp}')

In [ ]:
# Curvas ROC y Precision-Recall
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for res, color, label in [(res_rf, '#065A82', 'Random Forest'), (res_xgb, '#F96167', 'XGBoost')]:
    fpr, tpr, _ = roc_curve(y_test, res['_prob'])
    axes[0].plot(fpr, tpr, color=color, label=f"{label} (AUC={res['AUC-ROC']:.3f})")
    
    prec, rec, _ = precision_recall_curve(y_test, res['_prob'])
    axes[1].plot(rec, prec, color=color, label=f"{label} (F1={res['F1']:.3f})")

axes[0].plot([0,1],[0,1],'k--', linewidth=0.8)
axes[0].set(title='Curva ROC', xlabel='FPR', ylabel='TPR')
axes[0].legend()

axes[1].set(title='Curva Precision-Recall', xlabel='Recall', ylabel='Precision')
axes[1].axhline(y=0.02, color='gray', linestyle='--', linewidth=0.8, label='Baseline (2%)')
axes[1].legend()

plt.tight_layout()
plt.savefig('fig_curvas.png', dpi=120, bbox_inches='tight')
plt.show()

**Justificación de métricas:** En detección de fraude con fuerte desbalance, **Recall** (fraudes correctamente detectados) y **F1** son prioritarios sobre Accuracy, ya que un falso negativo (fraude no detectado) tiene mayor impacto financiero que un falso positivo (bloqueo innecesario).

---
## MÓDULO 5 — Interpretabilidad y Recomendaciones

In [ ]:
# Feature Importance del modelo final
# Extraemos el clasificador del pipeline
clf_final = mejor_modelo.named_steps['clf']
importances = pd.Series(clf_final.feature_importances_, index=features).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(8, 5))
importances.head(12).plot(kind='barh', ax=ax, color='#065A82')
ax.invert_yaxis()
ax.set_title(f'Top Variables — {mejor_res["Modelo"]}')
ax.set_xlabel('Importancia')
plt.tight_layout()
plt.savefig('fig_importance.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# Análisis SHAP — Valores Globales
# Tomamos una muestra para agilizar el cálculo
scaler_shap = mejor_modelo.named_steps['scaler']
X_test_scaled_shap = scaler_shap.transform(X_test)
X_test_scaled_df   = pd.DataFrame(X_test_scaled_shap, columns=features)

explainer = shap.TreeExplainer(clf_final)
shap_values = explainer.shap_values(X_test_scaled_df)

# Para clasificación binaria: tomamos la clase positiva (fraude)
sv = shap_values[1] if isinstance(shap_values, list) else shap_values

plt.figure(figsize=(9, 5))
shap.summary_plot(sv, X_test_scaled_df, plot_type='bar', show=False)
plt.title('SHAP — Importancia Global de Variables')
plt.tight_layout()
plt.savefig('fig_shap_global.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# SHAP — Explicación local: primera transacción predicha como fraude
y_prob_test = mejor_modelo.predict_proba(X_test)[:, 1]
fraude_idx  = np.where(y_prob_test >= 0.5)[0]

if len(fraude_idx) > 0:
    idx = fraude_idx[0]
    shap_exp = shap.Explanation(
        values        = sv[idx],
        base_values   = explainer.expected_value[1] if isinstance(explainer.expected_value, list) else explainer.expected_value,
        data          = X_test_scaled_df.iloc[idx].values,
        feature_names = features
    )
    plt.figure(figsize=(10, 4))
    shap.plots.waterfall(shap_exp, show=False)
    plt.title('SHAP Waterfall — Transacción predicha como Fraude')
    plt.tight_layout()
    plt.savefig('fig_shap_local.png', dpi=120, bbox_inches='tight')
    plt.show()
    print(f'Transacción índice {idx} — Probabilidad de fraude: {y_prob_test[idx]:.4f}')
else:
    print('No hay transacciones predichas como fraude con umbral 0.5 — reducir umbral.')

In [ ]:
# Análisis de umbral óptimo (Precision-Recall tradeoff)
prec_arr, rec_arr, thresholds = precision_recall_curve(y_test, y_prob_test)
f1_arr = 2 * prec_arr[:-1] * rec_arr[:-1] / (prec_arr[:-1] + rec_arr[:-1] + 1e-9)
umbral_optimo = thresholds[np.argmax(f1_arr)]

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(thresholds, prec_arr[:-1], label='Precision', color='#065A82')
ax.plot(thresholds, rec_arr[:-1],  label='Recall',    color='#F96167')
ax.plot(thresholds, f1_arr,        label='F1',        color='#028090')
ax.axvline(umbral_optimo, color='gray', linestyle='--', label=f'Umbral óptimo={umbral_optimo:.2f}')
ax.set(xlabel='Umbral de decisión', title='Análisis de Umbral — Precision vs Recall vs F1')
ax.legend()
plt.tight_layout()
plt.savefig('fig_umbral.png', dpi=120, bbox_inches='tight')
plt.show()
print(f'Umbral óptimo (max F1): {umbral_optimo:.4f}')

In [ ]:
# Evaluación con umbral óptimo
y_pred_opt = (y_prob_test >= umbral_optimo).astype(int)
print(f'=== Métricas con Umbral Óptimo ({umbral_optimo:.2f}) ===')
print(classification_report(y_test, y_pred_opt, target_names=['Legítima', 'Fraude']))

### Reflexiones finales

**Concept Drift:**
- Cambios en los patrones de comportamiento de los clientes (e.g., mayor uso digital post-pandemia) pueden degradar el modelo.
- Estrategias fraudulentas evolucionan continuamente; los patrones entrenados hoy pueden quedar obsoletos.
- Se recomienda monitoreo mensual del AUC-ROC y Recall sobre datos en producción.

**Próximos pasos:**
1. Implementar el modelo con el umbral óptimo identificado.
2. Establecer un pipeline de monitoreo continuo con alertas de degradación.
3. Reentrenar trimestralmente con nuevos datos etiquetados.
4. Evaluar la integración con el sistema de alertas de la mesa de control.